# Hebrew Law Corpus: Docling → LLooM (baseline / v1)

Extract Israeli bill PDFs using Docling, chunk them, and run LLooM concept discovery.

The deterministic preparation steps (PDF extraction + OCR fallback, legal chunking,
and LLooM model/session setup) now live in the shared, tested `lawsofisrael`
package under `src/`. This baseline notebook consumes those functions so the v1
and v2 experiments share one implementation. v1 keeps LLooM's **default** prompts
and still runs the score + export stage.

LLM calls go through an OpenAI-compatible API (Gemini 2.5 Flash by default - see `config.py`).
Embeddings stay local (bge-m3 via `scripts/start_embedding_server.sh`).

In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().resolve().parent if (Path.cwd().resolve() / 'dataset').exists() else Path.cwd().resolve().parent
# Make both the repo root (for config.py) and the src/ package importable.
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'src'))

from config import (
    MODEL_CONFIG, EMBEDDING_URL, EMBEDDING_MODEL, EMBEDDING_API_KEY,
    MAX_CONTEXT_TOKENS, MAX_OUTPUT_TOKENS,
)
from lawsofisrael import extraction, chunking
from lawsofisrael.lloom import make_lloom_session

CACHE = ROOT / 'notebooks' / 'cache'
EXTRACTION_CACHE = CACHE / 'extraction'
OUTPUTS = ROOT / 'notebooks' / 'outputs'

# Chat/LLM: OpenAI-compatible API. Provider, model, rate limits and pricing all
# come from config.py (driven by .env) - swap providers there, not here.
CHAT_MODEL = MODEL_CONFIG['name']
CHAT_API_KEY = MODEL_CONFIG['api_key']
CHAT_BASE_URL = MODEL_CONFIG['base_url']

# Embeddings: local llama.cpp server (scripts/start_embedding_server.sh).
EMBED_URL = EMBEDDING_URL
EMBED_MODEL = EMBEDDING_MODEL
EMBED_API_KEY = EMBEDDING_API_KEY

# Per-chunk token budget (approximated with cl100k_base inside the package),
# reserving prompt overhead + output headroom against the context window.
CHUNK_TOKENS = chunking.compute_chunk_budget(MAX_CONTEXT_TOKENS, MAX_OUTPUT_TOKENS)

# How many bills to process (None = all)
N_BILLS = 30

## 1. Verify Servers

In [ ]:
import requests
from openai import OpenAI

if not CHAT_API_KEY:
    print("✗ OPENAI_API_KEY is not set. Copy .env.example to .env and fill it in.")

try:
    requests.get(f"{EMBED_URL}/health", timeout=5).raise_for_status()
    print("✓ Embedding server running")
except Exception as e:
    print(f"✗ Embedding server not responding: {e}")
    print("  Start it with: ./scripts/start_embedding_server.sh")

chat = OpenAI(base_url=CHAT_BASE_URL, api_key=CHAT_API_KEY)
resp = chat.chat.completions.create(model=CHAT_MODEL, messages=[{"role": "user", "content": "השב בעברית: שלום"}], max_tokens=200)
print(f"✓ Chat API ({CHAT_MODEL}): {resp.choices[0].message.content}")

## 2. Load Dataset

In [3]:
from datasets import load_dataset

dataset = load_dataset('nickbes/lawsofisrael', split='train')
df = dataset.to_pandas()
if N_BILLS:
    df = df.head(N_BILLS)
print(f"Processing {len(df)} bills")

Processing 30 bills


## 3. Extract PDFs with Docling

Each bill cached immediately to `notebooks/cache/extraction/{bill_id}.json`. Resumable - skips already cached.

In [ ]:
# Extraction (Docling + selective Hebrew OCR fallback), the extraction-record
# schema, and incremental JSON caching now live in lawsofisrael.extraction.
# extraction.extract_bill(row, EXTRACTION_CACHE) reads a valid cached record if
# present (upgrading legacy records in memory) and otherwise extracts + caches.
def extract_bill(row):
    return extraction.extract_bill(row, EXTRACTION_CACHE)

In [5]:
# Extract bills (skips already cached)
print(f"Extracting {len(df)} bills...")
records = []
for i, (_, row) in enumerate(df.iterrows(), 1):
    record = extract_bill(row.to_dict())
    records.append(record)
    status = '✓' if record['status'] == 'success' else '✗'
    print(f"{status} {i}/{len(df)}: {record['bill_id']}")

success = sum(r['status'] == 'success' for r in records)
print(f"\nDone: {success}/{len(records)} successful")

Extracting 30 bills...
✓ 1/30: 1057303
✓ 2/30: 2229019
✓ 3/30: 2240475
✓ 4/30: 2240465
✓ 5/30: 1057405
✓ 6/30: 1057406
✓ 7/30: 2244713
✓ 8/30: 2244462
✓ 9/30: 2218723
✓ 10/30: 1042100
✓ 11/30: 2197909
✓ 12/30: 2168881
✓ 13/30: 2220910
✓ 14/30: 2202055
✓ 15/30: 1046237
✓ 16/30: 2204244
✓ 17/30: 2240508
✓ 18/30: 2219317
✓ 19/30: 2200908
✓ 20/30: 1046320
✓ 21/30: 1046300
✓ 22/30: 2244347
✓ 23/30: 2225649
✓ 24/30: 2206757
✓ 25/30: 2240494
✓ 26/30: 2229708
✓ 27/30: 2208737
✓ 28/30: 2228513
✓ 29/30: 2237751
✓ 30/30: 2221116

Done: 30/30 successful


## 4. Chunk Text

In [ ]:
# Legal-unit splitting, token budgeting, hard splitting, stable chunk IDs, and
# provenance-preserving source spans now live in lawsofisrael.chunking. Chunk
# text, IDs, token counts, and legacy heading_path/page_no match the original
# v1 behavior exactly; v2 additionally uses the per-chunk `source_spans`.
chunk_objs = chunking.chunk_records(records, CHUNK_TOKENS)
chunks_df = pd.DataFrame(chunking.chunks_to_records(chunk_objs))

# v1 works with the legacy columns; drop the v2-only provenance column here so
# the baseline stays identical to before.
chunks_df = chunks_df[['bill_id', 'name', 'heading_path', 'page_no', 'text', 'chunk_id', 'tokens']]

assert chunks_df['tokens'].max() <= CHUNK_TOKENS, 'chunk exceeds token budget'

print(f"Built {len(chunks_df)} chunks from {chunks_df['bill_id'].nunique()} bills")
print(f"{CHAT_MODEL} tokens/chunk (approx, cl100k_base): mean={chunks_df['tokens'].mean():.0f} "
      f"median={chunks_df['tokens'].median():.0f} max={chunks_df['tokens'].max()} "
      f"total={chunks_df['tokens'].sum():,}")
chunks_df.head()

## 5. LLooM Concept Discovery

In [ ]:
import builtins, contextlib

@contextlib.contextmanager
def auto_confirm():
    orig = builtins.input
    builtins.input = lambda _: 'y'
    try:
        yield
    finally:
        builtins.input = orig

# Model/session construction (including the non-OpenAI tokenizer compatibility
# shims LLooM needs for names like "gemini-2.5-flash") now lives in
# lawsofisrael.lloom.make_lloom_session.
def make_lloom(df):
    return make_lloom_session(
        df,
        model_config=MODEL_CONFIG,
        chat_base_url=CHAT_BASE_URL,
        chat_api_key=CHAT_API_KEY,
        max_output_tokens=MAX_OUTPUT_TOKENS,
        embed_url=EMBED_URL,
        embed_model_name=EMBED_MODEL,
        embed_api_key=EMBED_API_KEY,
    )

In [10]:
print(f"Running LLooM on {len(chunks_df)} chunks...")
session = make_lloom(chunks_df)

with auto_confirm():
    # Distill compresses each chunk to a fixed number of quotes/bullets.
    # For large chunks, increase these if you want more granular extraction.
    await session.gen(params={'filter_n_quotes': 5, 'summ_n_bullets': 7, 'synth_n_concepts': 3}, auto_review=True)

await session.select_auto(max_concepts=5)

concepts_df = pd.DataFrame([{'concept_id': k, **v.to_dict()} for k, v in session.concepts.items()])
concepts_df

Running LLooM on 30 chunks...


Estimated cost: $0.23
**Please note that this is only an approximate cost estimate**


Action required


Distill-filter
⠧ Loading ERROR json_load on: ```json
{
    "relevant_quotes": [
        "חוק לתיקון פקודת בתי הסוהר )הארכת הוראות שעה( )תיקוני חקיקה(, התשפ"ו2026",
        "חוק לתיקון פקודת בתי הסוהר )מס' 64 - הוראת שעה - חרבות ברזל( )מצב חירום כליאתי(, התשפ"ד2023- - מס' 5",
        "חוק לתיקון פקודת בתי הסוהר )תיקון מס' 66 - הוראת שעה - חרבות ברזל( )חופשה מיוחדת לאסיר(, התשפ"ד2024- - מס' 5",
        "הוראת מעבר . .3 הכרזה שניתנה לפני תחילתו של חוק זה לפי סעיף 19כ לפקודה, כנוסחו בהוראת השעה האמורה בסעיף 1 לחוק זה, מוארכת ותמשיך לעמוד בתוקפה עד יום ד' בתשרי התשפ"ז )15 בספטמבר 2026( .",
        "* התקבל בכנסת ביום י"ד באב התשפ"ו )28 ביולי 2026(; הצעת החוק ודברי הסבר פורסמו בהצעות חוק הממשלה -, ,1960 מיום י"ד בתמוז התשפ"ו )29 ביוני 2026(, עמ' . 1462"
    ]
}
```
ERROR json_load on: ```json
{
    "relevant_quotes": [
        "חוק לתיקון פקודת העיריות )מס'

,chunk_id,text
0,2229019:0000,תיקון חוק הליכי חקירה והעדה )התאמה לאנשים עם מ...
1,2240465:0000,ספר החוקים\nאמיר אוחנה יושב ראש הכנסת\nיצחק הר...
2,1057405:0000,חוק לתיקון ולהארכת תוקפן של תקנות שעת חירום )ח...
3,2244462:0000,מטרתו של פרק זה לקבוע הוראות מיוחדות הדרושות ל...
4,1042100:0000,מוקמת בזה הרשות לתקשורת משודרת .\nהרשות תהיה ע...
5,2202055:0000,"1 )להלן - החוק העיקרי(, בסעיף 18א)ב(, בהגדרה ""..."
6,1046237:0000,"בחוק איסור הונאה בכשרות, התשמ""ג-1983\nאלה רשאי..."
7,2204244:0000,חובת רישום במרשם\nחובת קבלת רישיון\nעיצום כספי...
8,2219317:0000,16ד. ד. )א( בעסקה מסוגי העסקאות המנויות בתוספת...
9,2200908:0000,"מטרתו של חוק זה להנציח את זכרה, פועלה ומורשתה ..."




Distill-summarize
✅ Done    


,chunk_id,text
0,2229019:0000,תיקון חוק הליכי חקירה והעדה
1,2229019:0000,התאמה לאנשים עם מוגבלות נפשית
2,2229019:0000,תיקון חוק הנוער שפיטה וענישה
3,2229019:0000,הוראת שעה מספר עשרות
4,2229019:0000,זכות לנוכחות עורך דין בחקירה
...,...,...
72,2228513:0000,תיקון מספר עשרים וארבע בחוק
73,2228513:0000,תיקון מספר שישים ושלוש בחוק
74,2228513:0000,חובת דיווח שוטפת אל הכנסת
75,2228513:0000,הוראת שעה זמנית ומחייבת כחוק




Cluster
✅ Done    


,chunk_id,text,cluster_id
13,2240465:0000,המקור הוא בהצעות חוק הממשלה,-1
14,1057405:0000,חוקים הקשורים לתקופות חירום ביטחוניות,-1
8,2240465:0000,אמיר אוחנה הוא יושב הכנסת,-1
31,1042100:0000,תפקיד הרשות הוא אסדרה ופיקוח,-1
25,2244462:0000,קיום ישיבות בהיוועדות חזותית מרחוק,-1
...,...,...,...
50,2204244:0000,דרישה לקבלת רישיון פעילות תקף,2
37,2202055:0000,הוראות תחולה על תביעות קיימות,2
45,1046237:0000,הרבנות הראשית תפעיל מערך פיקוח,2
74,2228513:0000,חובת דיווח שוטפת אל הכנסת,2




Synthesize
✅ Done    


Input examples: ['המקור הוא בהצעות חוק הממשלה', 'חוקים הקשורים לתקופות חירום ביטחוניות', 'אמיר אוחנה הוא יושב הכנסת', 'תפקיד הרשות הוא אסדרה ופיקוח', 'קיום ישיבות בהיוועדות חזותית מרחוק', 'תיקוני חקיקה לעת הגבלה ביטחונית', 'הסדרת מעמדם של עצורים ואסירים', 'קיום דיונים משפטיים בהיוועדות חזותית', 'השר רשאי להתקין תקנות לביצועו', 'חובה לנהל מרשם רשמי מוסדר', 'הטלת עיצום כספי בגין הפרה', 'הסדרת מנגנון תשלומים עבור פסולת', 'פיקוח על מחזור ופינוי פסולת', 'רשות כשרות מקומית תיתן השירותים', 'הוספת דודה ובנם או בתם', 'תביעות שטרם התיישנו לפי הדין', 'יש ליידע הצרכן שהשיחה מוקלטת', 'הגדרת חובות ליצרן פסולת הבניין', 'הקמת אתר ארכיון ומכון מחקר', 'הקמת מרכז מחקר על שמה', 'חוק אומנה לילדים משנת 2016']
Output concepts: 
	name: חקיקה ורגולציה ממשלתית
	prompt: Determine if the text example describes government legislation, bills, amendments to laws, or regulatory authorities and their formal duties.
	example_ids: ['2240465:0000', '1042100:0000']
	name: אכיפה ופיקוח סביבתי
	pro

,concept_id,id,name,prompt,example_ids,active,summary,seed
0,d7c1531b-17fc-4305-afcb-9e4f7b061573,d7c1531b-17fc-4305-afcb-9e4f7b061573,אכיפה ופיקוח סביבתי,Determine if the text example relates to envir...,[2204244:0000],True,None,None
1,831afdc5-ada5-456e-968a-f86f5b6c815f,831afdc5-ada5-456e-968a-f86f5b6c815f,זכויות עצורים וביטחון,"Determine if the text example concerns laws, l...",[1057405:0000],True,None,None
2,553bf460-220b-4e80-b56d-af68a7bca54f,553bf460-220b-4e80-b56d-af68a7bca54f,תחום החלת החוק,Determine whether the given text describes the...,"[1042100:0000, 1046237:0000]",False,None,None
3,873e6e4e-741a-44b9-b8f9-dfab39595eeb,873e6e4e-741a-44b9-b8f9-dfab39595eeb,ממשל וניהול ציבורي,Determine whether the given text describes gov...,"[2200908:0000, 2202055:0000]",False,None,None
4,42ca8742-47d4-4ec1-97b1-08958b61513f,42ca8742-47d4-4ec1-97b1-08958b61513f,שירותים חברתיים ורווחה,Determine whether the given text relates to ac...,"[2229019:0000, 2244462:0000]",True,None,None
5,ddb7383f-2b75-4ca0-9d6b-c6af2e4dd329,ddb7383f-2b75-4ca0-9d6b-c6af2e4dd329,חובות רגולטוריות וזכויות צרכנים,Determine whether the text example describes a...,"[2228513:0000, 2219317:0000]",True,None,None
6,df31d7d8-8bab-405f-87bc-491d743f299c,df31d7d8-8bab-405f-87bc-491d743f299c,הקמה וסמכויות של רשויות,Determine whether the text example relates to ...,"[1042100:0000, 1046237:0000]",False,None,None
7,de487083-6448-4a03-809e-9a6b32e548c8,de487083-6448-4a03-809e-9a6b32e548c8,הוראות מיוחדות לבחירות,Determine whether the text example outlines sp...,[2244462:0000],False,None,None
8,60643995-1cba-4d38-983e-b84680b1c37d,09c7c0bd-4d62-47e0-b458-3c99647ff662,חקיקה ופרסומים רשמיים,Determine if the text example discusses offici...,"[1042100:0000, 2240465:0000, 2240465:0000, 222...",False,None,None
9,5fdd039b-ca38-4be5-87be-4b38b0b7d5b3,2069b48d-78e0-4a85-843b-c6ad8e32711d,חקיקה ותיקוני חוק,Determine whether the text discusses governmen...,"[1042100:0000, 2240465:0000, 2229019:0000, 222...",False,None,None


## 6. Score & Export

In [11]:
# Score packs batch_size raw chunks plus concept criteria into one prompt.
# batch_size=1 keeps each request's worst case around CHUNK_TOKENS tokens plus
# concept criteria, which fits comfortably within MAX_CONTEXT_TOKENS. The
# truncate_fn will silently drop the tail of anything that overflows the
# configured context_window, which would score those chunks against nothing.
with auto_confirm():
    await session.score(df=chunks_df[['chunk_id', 'text']], batch_size=1, get_highlights=False)

scores_df = pd.concat(session.results.values(), ignore_index=True).rename(columns={'doc_id': 'chunk_id'})
scores_df = scores_df.merge(chunks_df[['chunk_id', 'bill_id', 'name']], on='chunk_id')

# Export
OUTPUTS.mkdir(parents=True, exist_ok=True)
concepts_df.to_parquet(OUTPUTS / 'concepts.parquet', index=False)
scores_df.to_parquet(OUTPUTS / 'scores.parquet', index=False)

print(f"Exported to {OUTPUTS}")
scores_df.head()



Scoring 4 concepts for 30 documents
Estimated cost: $0.24
**Please note that this is only an approximate cost estimate**


Action required
100%|██████████| 4/4 [03:06<00:00, 46.63s/it]
✅ Done with concept scoring!
Exported to /home/nick/Documents/projects/lawsofisrael/notebooks/outputs


,chunk_id,text,concept_id,concept_name,concept_prompt,score,rationale,highlight,concept_seed,bill_id,name
0,1057303:0000,"ט""ו באב התשפ""ו 3575 29 ביולי 2026\n\nעמוד\n\nח...",d7c1531b-17fc-4305-afcb-9e4f7b061573,אכיפה ופיקוח סביבתי,Determine if the text example relates to envir...,0.00,The text relates to amending the Prisons Ordin...,,None,1057303,חוק לתיקון פקודת בתי הסוהר (הארכת הוראות שעה) ...
1,2229019:0000,"ט""ו באב התשפ""ו 3578 29 ביולי 2026\n\nעמוד\n\nס...",d7c1531b-17fc-4305-afcb-9e4f7b061573,אכיפה ופיקוח סביבתי,Determine if the text example relates to envir...,0.00,The text is an Israeli legislative amendment c...,,None,2229019,חוק נוכחות עורך דין בחקירת קטינים ואנשים עם מו...
2,2240475:0000,"ט""ו באב התשפ""ו 3577 29 ביולי 2026\n\nעמוד\n\nח...",d7c1531b-17fc-4305-afcb-9e4f7b061573,אכיפה ופיקוח סביבתי,Determine if the text example relates to envir...,0.00,The text is a legislative amendment dealing wi...,,None,2240475,"חוק לתיקון פקודת העיריות (מס' 163), התשפ""ו–2026"
3,2240465:0000,"ספר החוקים\n\nעמוד\n\n)1( בפסקה )1(, ברישה, אח...",d7c1531b-17fc-4305-afcb-9e4f7b061573,אכיפה ופיקוח סביבתי,Determine if the text example relates to envir...,0.75,The text mentions the Committee for the Protec...,,None,2240465,"חוק התכנון והבנייה (תיקון מס' 170), התשפ""ו-2026"
4,1057405:0000,. .1 בחוק לתיקון ולהארכת תוקפן של תקנות שעת חי...,d7c1531b-17fc-4305-afcb-9e4f7b061573,אכיפה ופיקוח סביבתי,Determine if the text example relates to envir...,0.00,The text is an Israeli legislative amendment c...,,None,1057405,חוק לתיקון ולהארכת תוקפן של תקנות שעת חירום (ח...
